# Themenanalyse kommunaler Bürgerhinweise

Lokaler Prototyp zur Analyse von Meldetexten des Mängelmelders Konstanz mit LDA und NMF. Das Notebook dient als Grundlage für die weitere Entwicklung.

## Offene Punkte

- Die drei Varianten der Vorverarbeitung zusammenführen und die Textfilter prüfen.
- Verschiedene Vorgaben für die Anzahl der Topics systematisch vergleichen.
- Die Behandlung von Fehlwerten, Dubletten und Meldungen ohne verwertbare Begriffe prüfen.

Die Namen und Beschreibungen der Themen sowie die Beispiele sind im Code vorgegeben. Nach Änderungen am Modell müssen sie erneut überprüft werden.

Die Quelldatei `maengelmelder_konstanz_final_9_neu.csv` wird unter `data/raw/` erwartet. Die Zellen sind von oben nach unten auszuführen.


In [ ]:
# Schritt 01 Umgebung und Arbeitsordner prüfen

from pathlib import Path
import sys
import re

import pandas as pd
import numpy as np
import sklearn

print("Python executable:")
print(sys.executable)

print("\nPandas Version:")
print(pd.__version__)

print("\nNumPy Version:")
print(np.__version__)

print("\nScikit Learn Version:")
print(sklearn.__version__)

print("\nAktueller Arbeitsordner:")
print(Path.cwd())


In [ ]:
# Schritt 02 CSV-Datei laden

from pathlib import Path
import pandas as pd

CSV_NAME = "maengelmelder_konstanz_final_9_neu.csv"

possible_paths = [
    Path("data") / "raw" / CSV_NAME,
    Path("..") / "data" / "raw" / CSV_NAME,
]

csv_path = None

for path in possible_paths:
    print("Prüfe:", path.resolve())
    if path.exists():
        csv_path = path
        break

if csv_path is None:
    raise FileNotFoundError(
        "CSV wurde nicht gefunden. Prüfe, ob die Datei im Ordner data/raw liegt."
    )

print("\nGefundene CSV:")
print(csv_path.resolve())

df = pd.read_csv(csv_path)

print("\nShape:")
print(df.shape)

print("\nSpalten:")
print(list(df.columns))

df.head(3)


In [ ]:
# Schritt 03 Datenqualität prüfen

required_columns = [
    "id",
    "date_created",
    "text_raw",
    "category_parent",
    "category_child",
    "responsible",
    "lat",
    "lon",
    "address_input",
    "detail_url",
]

missing_required = [col for col in required_columns if col not in df.columns]

if missing_required:
    raise ValueError(f"Folgende erwartete Spalten fehlen: {missing_required}")

print("Anzahl Zeilen:", len(df))

print("\nFehlende Werte je Spalte:")
print(df.isna().sum())

print("\nFehlende Werte in text_raw:")
print(df["text_raw"].isna().sum())

print("\nDubletten in text_raw:")
print(df["text_raw"].duplicated().sum())

print("\nKategorien in category_parent:")
print(df["category_parent"].value_counts(dropna=False))

print("\nKategorien in category_child:")
print(df["category_child"].value_counts(dropna=False).head(25))

print("\nTextlängen in Wörtern:")
word_counts_raw = df["text_raw"].dropna().astype(str).str.split().str.len()
print(word_counts_raw.describe())

print("\nBeispiele aus text_raw:")
for i, text in enumerate(df["text_raw"].dropna().astype(str).head(5), start=1):
    print(f"\nBeispiel {i}:")
    print(text[:500])


In [ ]:
# Schritt 04 Arbeitsdatensatz erstellen

# Die Behandlung echter Fehlwerte vor der String-Umwandlung ist noch zu prüfen.
df_work = df.copy()

# Textwerte in Zeichenketten umwandeln und äußere Leerzeichen entfernen.
df_work["text_raw"] = df_work["text_raw"].astype(str).str.strip()

# Nach der Umwandlung leere Zeichenketten entfernen.
before_empty = len(df_work)
df_work = df_work[df_work["text_raw"].str.len() > 0].copy()
after_empty = len(df_work)

# Identische Meldetexte nach dem Entfernen äußerer Leerzeichen zusammenführen.
before_duplicates = len(df_work)
df_work = df_work.drop_duplicates(subset=["text_raw"]).copy()
after_duplicates = len(df_work)

# Wortanzahl im Rohtext ergänzen
df_work["word_count_raw"] = df_work["text_raw"].str.split().str.len()

# Meldungen mit weniger als zwei Wörtern ausschließen.
# Bei Bürgerhinweisen sind kurze Texte oft trotzdem aussagekräftig.
before_short_filter = len(df_work)
df_work = df_work[df_work["word_count_raw"] >= 2].copy()
after_short_filter = len(df_work)

df_work = df_work.reset_index(drop=True)

print("Ausgangszeilen:", len(df))
print("Nach Entfernen leerer Texte:", after_empty)
print("Entfernte leere Texte:", before_empty - after_empty)
print("Nach Entfernen von Dubletten:", after_duplicates)
print("Entfernte Dubletten:", before_duplicates - after_duplicates)
print("Nach Entfernen extrem kurzer Texte:", after_short_filter)
print("Entfernte extrem kurze Texte:", before_short_filter - after_short_filter)

print("\nVerteilung der Wortanzahl nach Filter:")
print(df_work["word_count_raw"].describe())

df_work[["text_raw", "category_parent", "category_child", "word_count_raw"]].head(10)


In [ ]:
# Schritt 05 Ausgabeordner festlegen

from pathlib import Path

OUTPUT_DIR = Path("outputs")

if not OUTPUT_DIR.exists():
    OUTPUT_DIR = Path("..") / "outputs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output-Ordner:")
print(OUTPUT_DIR.resolve())


In [ ]:
# Schritt 06 Textbereinigung im ursprünglichen Versuch

GERMAN_STOPWORDS = {
    "aber", "alle", "allem", "allen", "aller", "alles", "als", "also", "am", "an",
    "andere", "anderem", "anderen", "anderer", "anderes", "auch", "auf", "aus",
    "bei", "bin", "bis", "bist", "da", "damit", "dann", "der", "den", "des",
    "dem", "die", "das", "dass", "daß", "dazu", "dein", "deine", "deinem",
    "deinen", "deiner", "deines", "denn", "dich", "dir", "du", "dies", "diese",
    "diesem", "diesen", "dieser", "dieses", "doch", "dort", "durch", "ein",
    "eine", "einem", "einen", "einer", "eines", "einige", "einigen", "einiger",
    "einiges", "einmal", "er", "ihn", "ihm", "es", "etwas", "euer", "eure",
    "eurem", "euren", "eurer", "eures", "für", "gegen", "gewesen", "hab",
    "habe", "haben", "hat", "hatte", "hatten", "hier", "hin", "hinter", "ich",
    "mich", "mir", "ihr", "ihre", "ihrem", "ihren", "ihrer", "ihres", "euch",
    "im", "in", "indem", "ins", "ist", "jede", "jedem", "jeden", "jeder",
    "jedes", "jene", "jenem", "jenen", "jener", "jenes", "jetzt", "kann",
    "kein", "keine", "keinem", "keinen", "keiner", "keines", "können", "könnte",
    "machen", "man", "manche", "manchem", "manchen", "mancher", "manches",
    "mein", "meine", "meinem", "meinen", "meiner", "meines", "mit", "muss",
    "musste", "nach", "nicht", "nichts", "noch", "nun", "nur", "ob", "oder",
    "ohne", "sehr", "sein", "seine", "seinem", "seinen", "seiner", "seines",
    "selbst", "sich", "sie", "ihnen", "sind", "so", "solche", "solchem",
    "solchen", "solcher", "solches", "soll", "sollte", "sondern", "sonst",
    "über", "um", "und", "uns", "unser", "unsere", "unserem", "unseren",
    "unserer", "unseres", "unter", "viel", "vom", "von", "vor", "während",
    "war", "waren", "warst", "was", "weg", "weil", "weiter", "welche",
    "welchem", "welchen", "welcher", "welches", "wenn", "werde", "werden",
    "wie", "wieder", "will", "wir", "wird", "wirst", "wo", "wollen", "wollte",
    "würde", "würden", "zu", "zum", "zur", "zwar", "zwischen"
}

DOMAIN_STOPWORDS = {
    "bitte", "danke", "hallo", "ca", "circa", "seit", "immer", "mehrfach",
    "mehr", "direkt", "konstanz", "adresssuche", "autom", "ermittelt",
    "nr", "nummer", "siehe", "foto", "fotos"
}

STOPWORDS = GERMAN_STOPWORDS.union(DOMAIN_STOPWORDS)

def normalize_german_variants(text: str) -> str:
    replacements = {
        "muell": "müll",
        "grobmuell": "grobmüll",
        "strasse": "straße",
        "strassen": "straßen",
        "fussweg": "fußweg",
        "fusswege": "fußwege",
        "fussgaenger": "fußgänger",
        "ueber": "über",
        "oeffentlich": "öffentlich",
        "oeffentliche": "öffentliche",
        "umgestuerzt": "umgestürzt",
        "beschaedigt": "beschädigt",
        "verunreinigt": "verschmutzt",
        "verschmutzte": "verschmutzt"
    }

    for old, new in replacements.items():
        text = re.sub(rf"\b{old}\b", new, text)

    return text

def clean_text(text: str) -> str:
    text = str(text).lower()
    text = normalize_german_variants(text)

    # URLs und HTML entfernen
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)

    # Zahlen und Sonderzeichen entfernen, deutsche Umlaute behalten
    text = re.sub(r"[^a-zäöüß\s]", " ", text)

    # Mehrfache Leerzeichen entfernen
    text = re.sub(r"\s+", " ", text).strip()

    # Tokenisieren und Stopwörter entfernen
    tokens = text.split()
    tokens = [
        token for token in tokens
        if token not in STOPWORDS and len(token) >= 2
    ]

    return " ".join(tokens)

df_model = df_work.copy()
df_model["text_clean"] = df_model["text_raw"].apply(clean_text)
df_model["word_count_clean"] = df_model["text_clean"].str.split().str.len()

# Texte mit weniger als zwei verbleibenden Tokens ausschließen.
before_clean_filter = len(df_model)
df_model = df_model[df_model["word_count_clean"] >= 2].copy()
after_clean_filter = len(df_model)

df_model = df_model.reset_index(drop=True)

print("Zeilen vor Cleaning-Filter:", before_clean_filter)
print("Zeilen nach Cleaning-Filter:", after_clean_filter)
print("Entfernte Zeilen nach Cleaning:", before_clean_filter - after_clean_filter)

print("\nVerteilung der Wortanzahl nach Cleaning:")
print(df_model["word_count_clean"].describe())


In [ ]:
# Schritt 07 Textbereinigung mit erweiterten Stopwörtern und Begriffszuordnungen

EXTRA_STOPWORDS = {
    # Zusätzliche Begriffe, die in diesem Versuch als Stopwörter behandelt werden.
    "bitte", "danke", "hallo", "guten", "tag", "mfg", "ca", "circa", "etc",
    "seit", "immer", "mehrfach", "mehr", "schon", "heute", "gestern",
    "direkt", "bereich", "stelle", "ort", "ecke", "seite", "beiden",
    "gegenüber", "gegenueber", "neben", "vorne", "hinten", "oben", "unten",
    "befindet", "befinden", "kommt", "kommen", "geht", "gehen",
    "steht", "stehen", "liegt", "liegen", "wurde", "wurden", "worden",
    "str", "nr", "nummer", "haus", "höhe", "hoehe", "richtung",
    "nahe", "naehe", "konstanz", "adresssuche", "autom", "ermittelt",
    "siehe", "foto", "fotos", "bild", "bilder",

    # Zeitangaben
    "tagen", "tag", "tage", "wochen", "woche", "monaten", "monat",
    "jahren", "jahr", "nacht", "morgen", "abends", "mittags",
    "freitag", "samstag", "sonntag", "montag", "dienstag", "mittwoch", "donnerstag",

    # Weitere im bisherigen Versuch ausgeschlossene Begriffe zum Zustand von Objekten.
    "defekt", "defekte", "defekter", "defekten", "kaputt", "kaputte", "kaputtes",
    "beschädigt", "beschaedigt", "ausgefallen", "funktioniert", "funktion",
    "brennt", "brennen", "leuchtet", "leuchten", "leuchte", "dunkel",
    "schwach", "gemeldet", "bereits", "laengerem", "mehreren", "mal",
    "haengt", "hängt", "schaden", "schaeden", "schadhaft"
}

CANONICAL_TERMS = {
    # Ausgewählte Begriffe für Beleuchtung zusammenfassen.
    "strassenlaterne": "beleuchtung",
    "strassenlaternen": "beleuchtung",
    "straßenlaterne": "beleuchtung",
    "straßenlaternen": "beleuchtung",
    "strassenbeleuchtung": "beleuchtung",
    "straßenbeleuchtung": "beleuchtung",
    "strassenlampe": "beleuchtung",
    "strassenlampen": "beleuchtung",
    "strassenleuchte": "beleuchtung",
    "strassenleuchten": "beleuchtung",
    "laterne": "beleuchtung",
    "laternen": "beleuchtung",
    "lampe": "beleuchtung",
    "lampen": "beleuchtung",

    # Ausgewählte Schreibweisen und Begriffe zu Abfällen vereinheitlichen.
    "muell": "müll",
    "grobmuell": "grobmüll",
    "muelleimer": "mülleimer",
    "abfall": "müll",
    "abfälle": "müll",
    "abfaelle": "müll",
    "mülltonne": "müll",
    "mülltonnen": "müll",
    "müllcontainer": "müll",
    "container": "müllcontainer",

    # Straßen, Wege, Radverkehr
    "strasse": "straße",
    "strassen": "straße",
    "strassensteiten": "straßenseiten",
    "strassenseiten": "straßenseiten",
    "fussweg": "gehweg",
    "fusswege": "gehweg",
    "fußweg": "gehweg",
    "fußwege": "gehweg",
    "gehwege": "gehweg",
    "fussgaenger": "fußgänger",
    "fussgaengerampel": "ampel",
    "fußgängerampel": "ampel",
    "radwege": "radweg",
    "fahrräder": "fahrrad",
    "fahrraeder": "fahrrad",
    "fahrrads": "fahrrad",

    # Pflanzenwuchs
    "hecken": "hecke",
    "heckenwuchs": "hecke",
    "straeucher": "sträucher",
    "strauch": "sträucher",
    "baeume": "baum",
    "bäume": "baum",
    "aeste": "äste",

    # Schilder und sonstige Varianten
    "schilder": "schild",
    "verkehrsschild": "schild",
    "verkehrsschilder": "schild",
    "gullideckel": "gulli",
    "gullis": "gulli",
    "ueberfuellt": "überfüllt",
    "verunreinigt": "verschmutzt",
    "verschmutzte": "verschmutzt",
    "verschmutzten": "verschmutzt",
    "umgestuerzt": "umgestürzt",
    "zerstoert": "zerstört"
}

STOPWORDS_FINAL = GERMAN_STOPWORDS.union(EXTRA_STOPWORDS)

def clean_text_v2(text: str) -> str:
    text = str(text).lower()

    # URLs und HTML entfernen
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)

    # Zahlen und Sonderzeichen entfernen, Umlaute behalten
    text = re.sub(r"[^a-zäöüß\s]", " ", text)

    # Leerzeichen normalisieren
    text = re.sub(r"\s+", " ", text).strip()

    tokens = []
    for token in text.split():
        token = CANONICAL_TERMS.get(token, token)

        if token not in STOPWORDS_FINAL and len(token) >= 2:
            tokens.append(token)

    return " ".join(tokens)

df_model = df_work.copy()
df_model["text_clean"] = df_model["text_raw"].apply(clean_text_v2)
df_model["word_count_clean"] = df_model["text_clean"].str.split().str.len()

before_clean_filter = len(df_model)
df_model = df_model[df_model["word_count_clean"] >= 2].copy()
after_clean_filter = len(df_model)

df_model = df_model.reset_index(drop=True)

print("Zeilen vor Cleaning-Filter:", before_clean_filter)
print("Zeilen nach Cleaning-Filter:", after_clean_filter)
print("Entfernte Zeilen nach Cleaning:", before_clean_filter - after_clean_filter)

print("\nVerteilung der Wortanzahl nach Cleaning:")
print(df_model["word_count_clean"].describe())

print("\nBeispiele vorher/nachher:")
for _, row in df_model[["text_raw", "text_clean"]].head(10).iterrows():
    print("\nRAW:")
    print(row["text_raw"][:300])
    print("CLEAN:")
    print(row["text_clean"])


In [ ]:
# Schritt 08 Textbereinigung mit weiteren Ergänzungen

SUPPLEMENT_STOPWORDS = {
    # übrig gebliebene Schreibvarianten deutscher Füllwörter
    "fuer", "ueber", "waere", "waeren", "koennen", "koennte",
    "moeglich", "moegliche", "moeglicherweise", "wuerde", "wuerden",

    # Zusätzliche Begriffe aus der bisherigen Sichtung der Topics.
    "dank", "vielen", "viele", "stadt", "beim", "kaum", "stark",
    "aktuell", "allerdings", "zwei", "ganze", "andere", "anderen",
    "weiteres", "teile", "teil", "alte", "alter", "altes",
    "mehrere", "mindestens", "komplett", "vermutlich", "leider",
    "zeit", "sieht", "hausnummer",

    # Weitere im bisherigen Versuch ausgeschlossene Verben und Zustandsangaben.
    "abgestellt", "abgestellte", "abgestellter", "abgestelltes",
    "abgelegt", "entsorgt", "entfernen", "abgeholt", "abholung",
    "abgelaufen", "abgeschlossen",

    # häufige Orts- und Straßennamen aus der ersten Auswertung
    "beethovenstr", "sonnentauweg", "peter", "thumb", "litzelstetten",
    "mainaustrasse", "max", "heugaessle", "sigrist", "frieda",
    "hoehenweg", "steinstrasse", "flurweg", "dorffriedhofsweg",
    "hans", "hansegartenstrasse", "stromeyer", "mannheimer",
    "kreuzlinger", "macairestrasse", "schneckenburgstrasse",
    "urisberg", "allmannsdorf", "allmannshoehe",

    "wäre", "wären", "können", "könnte", "möglich",
    "unteren", "untere", "konstanzer",
    "schoen", "schön", "dankeschoen", "dankeschön",
    "fuss"
}

CANONICAL_TERMS_UPDATE = {
    "fuer": "für",
    "ueber": "über",
    "waechst": "wächst",
    "gefaehrlich": "gefährlich",
    "gruen": "grün",
    "moeglich": "möglich",
    "koennen": "können",
    "waere": "wäre",
    "beschaedigte": "beschädigt",
    "beschaedigten": "beschädigt",
    "beschädigte": "beschädigt",
    "haengt": "hängt",
    "laeuft": "läuft",
    "ueberfuellt": "überfüllt"
}

STOPWORDS_FINAL = STOPWORDS_FINAL.union(SUPPLEMENT_STOPWORDS)
CANONICAL_TERMS_FINAL = {**CANONICAL_TERMS, **CANONICAL_TERMS_UPDATE}

# Die Endungsregeln sind eine Heuristik und erkennen Ortsangaben nicht vollständig.
def is_location_token(token: str) -> bool:
    # Mögliche Straßennamen anhand der Endung erkennen und "straße" ausnehmen.
    if token == "straße":
        return False

    if token.endswith(("strasse", "straße", "str")) and len(token) > 5:
        return True

    # Mögliche Wegnamen erkennen und die folgenden allgemeinen Wegbegriffe ausnehmen.
    if token in {"gehweg", "radweg", "fahrradweg"}:
        return False

    if token.endswith("weg") and len(token) > 5:
        return True

    return False

def clean_text_v3(text: str) -> str:
    text = str(text).lower()

    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^a-zäöüß\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = []

    for token in text.split():
        token = CANONICAL_TERMS_FINAL.get(token, token)

        if is_location_token(token):
            continue

        if token not in STOPWORDS_FINAL and len(token) >= 2:
            tokens.append(token)

    return " ".join(tokens)

df_model = df_work.copy()
df_model["text_clean"] = df_model["text_raw"].apply(clean_text_v3)
df_model["word_count_clean"] = df_model["text_clean"].str.split().str.len()

before_clean_filter = len(df_model)
df_model = df_model[df_model["word_count_clean"] >= 2].copy()
after_clean_filter = len(df_model)

df_model = df_model.reset_index(drop=True)

print("Zeilen vor Cleaning-Filter:", before_clean_filter)
print("Zeilen nach Cleaning-Filter:", after_clean_filter)
print("Entfernte Zeilen nach Cleaning:", before_clean_filter - after_clean_filter)

print("\nVerteilung der Wortanzahl nach Cleaning:")
print(df_model["word_count_clean"].describe())

print("\nBeispiele vorher/nachher:")
for _, row in df_model[["text_raw", "text_clean"]].head(10).iterrows():
    print("\nRAW:")
    print(row["text_raw"][:250])
    print("CLEAN:")
    print(row["text_clean"])


In [ ]:
# Schritt 09 Bereinigte Texte anhand von Beispielen ansehen

print("Beispiele vorher/nachher:")

for _, row in df_model[["text_raw", "text_clean"]].head(10).iterrows():
    print("\nRAW:")
    print(row["text_raw"][:300])
    print("CLEAN:")
    print(row["text_clean"])


In [ ]:
# Schritt 10 Texte mit CountVectorizer und TfidfVectorizer vektorisieren

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

texts_clean = df_model["text_clean"].tolist()

count_vectorizer = CountVectorizer(
    min_df=5,
    max_df=0.90,
    token_pattern=r"(?u)\b[a-zäöüß][a-zäöüß]+\b",
    ngram_range=(1, 1)
)

tfidf_vectorizer = TfidfVectorizer(
    min_df=5,
    max_df=0.90,
    token_pattern=r"(?u)\b[a-zäöüß][a-zäöüß]+\b",
    ngram_range=(1, 1),
    sublinear_tf=True
)

X_count = count_vectorizer.fit_transform(texts_clean)
X_tfidf = tfidf_vectorizer.fit_transform(texts_clean)

print("Anzahl Dokumente:", len(texts_clean))
print("Count-Matrix:", X_count.shape)
print("TF-IDF-Matrix:", X_tfidf.shape)

print("\nBeispiele aus Count-Vokabular:")
print(count_vectorizer.get_feature_names_out()[:30])

print("\nBeispiele aus TF-IDF-Vokabular:")
print(tfidf_vectorizer.get_feature_names_out()[:30])


In [ ]:
# Schritt 11 LDA und NMF trainieren

from sklearn.decomposition import LatentDirichletAllocation, NMF

# Bisherige Einstellung. Die systematische Auswahl der Themenanzahl steht noch aus.
N_TOPICS = 10

lda_model = LatentDirichletAllocation(
    n_components=N_TOPICS,
    random_state=42,
    learning_method="batch",
    max_iter=20
)

nmf_model = NMF(
    n_components=N_TOPICS,
    random_state=42,
    init="nndsvda",
    max_iter=500
)

lda_doc_topic = lda_model.fit_transform(X_count)
nmf_doc_topic = nmf_model.fit_transform(X_tfidf)

print("LDA Dokument-Topic-Matrix:", lda_doc_topic.shape)
print("NMF Dokument-Topic-Matrix:", nmf_doc_topic.shape)


In [ ]:
# Schritt 12 Wichtigste Begriffe je Topic anzeigen

def print_top_words(model, feature_names, n_top_words=12, title=""):
    print("\n" + title)
    print("=" * len(title))

    for topic_idx, topic in enumerate(model.components_):
        top_indices = topic.argsort()[::-1][:n_top_words]
        top_words = [feature_names[i] for i in top_indices]
        print(f"Topic {topic_idx + 1}: {', '.join(top_words)}")

print_top_words(
    lda_model,
    count_vectorizer.get_feature_names_out(),
    n_top_words=12,
    title="LDA Topics auf CountVectorizer"
)

print_top_words(
    nmf_model,
    tfidf_vectorizer.get_feature_names_out(),
    n_top_words=12,
    title="NMF Topics auf TfidfVectorizer"
)


In [ ]:
# Schritt 13 Technische Themenübersichten erstellen

def build_topic_summary(model, feature_names, doc_topic_matrix, source_df, n_top_words=12, n_examples=2):
    # Bei mehreren Topics mit demselben höchsten Gewicht wird das erste gewählt.
    # Meldungen, deren Topic-Gewichte alle 0 sind, werden noch nicht gesondert behandelt.
    dominant_topic = doc_topic_matrix.argmax(axis=1)
    n_docs = len(source_df)

    rows = []

    for topic_idx in range(model.n_components):
        topic_number = topic_idx + 1

        # Top Begriffe
        topic_weights = model.components_[topic_idx]
        top_indices = topic_weights.argsort()[::-1][:n_top_words]
        top_words = ", ".join(feature_names[top_indices])

        # Dokumente mit dominantem Topic
        doc_indices = np.where(dominant_topic == topic_idx)[0]
        doc_count = len(doc_indices)
        doc_share = round((doc_count / n_docs) * 100, 1)

        # Häufigste bestehende Kategorie zur Plausibilisierung
        if doc_count > 0:
            category_counts = (
                source_df.iloc[doc_indices]["category_parent"]
                .value_counts()
                .head(3)
            )
            top_categories = " | ".join(
                [f"{cat}: {count}" for cat, count in category_counts.items()]
            )

            # Erste zugeordnete Meldungen auswählen und kürzen.
            examples = " || ".join(
                source_df.iloc[doc_indices]["text_raw"]
                .astype(str)
                .head(n_examples)
                .str.slice(0, 180)
                .tolist()
            )
        else:
            top_categories = ""
            examples = ""

        rows.append({
            "Topic": topic_number,
            "Anzahl Meldungen": doc_count,
            "Anteil Prozent": doc_share,
            "Top Begriffe": top_words,
            "Häufigste Kategorie": top_categories,
            "Beispielmeldungen": examples
        })

    summary = pd.DataFrame(rows)
    summary = summary.sort_values("Anzahl Meldungen", ascending=False).reset_index(drop=True)
    summary.insert(0, "Rang", np.arange(1, len(summary) + 1))

    return summary

lda_summary = build_topic_summary(
    lda_model,
    count_vectorizer.get_feature_names_out(),
    lda_doc_topic,
    df_model
)

nmf_summary = build_topic_summary(
    nmf_model,
    tfidf_vectorizer.get_feature_names_out(),
    nmf_doc_topic,
    df_model
)

print("LDA Themenübersicht:")
display(lda_summary)

print("NMF Themenübersicht:")
display(nmf_summary)


In [ ]:
# Schritt 14 Technische Themenübersichten als CSV speichern

OUTPUT_DIR = Path("outputs")

if not OUTPUT_DIR.exists():
    OUTPUT_DIR = Path("..") / "outputs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Die Spalte mit Rohtextbeispielen nicht in diesen CSV-Export übernehmen.
# Beispielmeldungen können Adressen oder personenbezogene Hinweise enthalten.
lda_export = lda_summary.drop(columns=["Beispielmeldungen"], errors="ignore")
nmf_export = nmf_summary.drop(columns=["Beispielmeldungen"], errors="ignore")

lda_path = OUTPUT_DIR / "themenuebersicht_lda.csv"
nmf_path = OUTPUT_DIR / "themenuebersicht_nmf.csv"

lda_export.to_csv(lda_path, index=False, encoding="utf-8-sig")
nmf_export.to_csv(nmf_path, index=False, encoding="utf-8-sig")

print("Gespeichert:")
print(lda_path.resolve())
print(nmf_path.resolve())


In [ ]:
# Schritt 15 NMF-Ergebnisse mit Themenbeschreibungen ergänzen

from sklearn.feature_extraction.text import CountVectorizer
from pathlib import Path
import numpy as np
import pandas as pd

# Dominantes NMF-Topic je Meldung bestimmen
df_topic_docs = df_model.copy()
df_topic_docs["NMF_Topic"] = nmf_doc_topic.argmax(axis=1) + 1

# Manuell formulierte Themenbezeichnungen, Beschreibungen und beispielhafte Hinweise.
# Die Zuordnung gilt nur für den bisherigen Modellstand mit zehn Topics.
# Nach erneutem Training ist die inhaltliche Zuordnung der Labels zu prüfen.
# Die beispielhaften Hinweise sind keine wörtlichen Originalmeldungen.
topic_meta = {
    1: {
        "Beschwerdethema": "Defekte Beleuchtung",
        "Kurzbeschreibung": "Viele Meldungen betreffen ausgefallene, flackernde oder schlecht sichtbare Beleuchtung im öffentlichen Raum.",
        "Typische Hinweise": "Laterne ausgefallen, Licht flackert, dunkler Bereich"
    },
    2: {
        "Beschwerdethema": "Herrenlose Fahrräder",
        "Kurzbeschreibung": "Viele Meldungen betreffen Fahrräder, die längere Zeit unbewegt stehen, beschädigt sind oder öffentliche Flächen blockieren.",
        "Typische Hinweise": "Fahrrad steht seit Wochen, Sattel fehlt, Fahrrad blockiert Platz"
    },
    3: {
        "Beschwerdethema": "Müll und Sauberkeit",
        "Kurzbeschreibung": "Viele Hinweise betreffen illegal abgelegten Müll, überfüllte Mülleimer und Verschmutzungen im öffentlichen Raum.",
        "Typische Hinweise": "Müll abgeladen, Mülleimer voll, Abfall im Gebüsch"
    },
    4: {
        "Beschwerdethema": "Gehwege und Heckenwuchs",
        "Kurzbeschreibung": "Meldungen betreffen zugewachsene Gehwege, Hecken, Engstellen und Einschränkungen für Fußgänger.",
        "Typische Hinweise": "Hecke wächst in Gehweg, Gehweg zu eng, Weg zugewachsen"
    },
    5: {
        "Beschwerdethema": "Straßenraum, Fahrzeuge und Behinderungen",
        "Kurzbeschreibung": "Das Thema bündelt Meldungen zu Straßen, Einfahrten, Parkplätzen, abgestellten Fahrzeugen und Behinderungen im Verkehrsraum.",
        "Typische Hinweise": "Fahrzeug abgestellt, Straße blockiert, Einfahrt beeinträchtigt"
    },
    6: {
        "Beschwerdethema": "Beschilderung",
        "Kurzbeschreibung": "Dieses Thema umfasst beschädigte, verschmutzte, umgefahrene oder schlecht erkennbare Schilder.",
        "Typische Hinweise": "Schild umgestürzt, Verkehrsschild verschmutzt, Schild schlecht sichtbar"
    },
    7: {
        "Beschwerdethema": "Radwege, Entwässerung und Gefahrenstellen",
        "Kurzbeschreibung": "Mehrere Meldungen betreffen Radwege und Wege mit Wasserstellen, verstopften Abläufen, Scherben oder anderen Gefahrenstellen.",
        "Typische Hinweise": "Wasser auf Radweg, Gulli verstopft, Scherben auf Weg"
    },
    8: {
        "Beschwerdethema": "Bäume, Äste und sonstige Hindernisse",
        "Kurzbeschreibung": "Meldungen beschreiben umgestürzte Bäume, überhängende Äste, Bewuchs oder Gegenstände, die Wege und Flächen blockieren.",
        "Typische Hinweise": "Baum umgestürzt, Äste hängen tief, Weg blockiert"
    },
    9: {
        "Beschwerdethema": "Ampeln und Fußgängerquerungen",
        "Kurzbeschreibung": "Meldungen beziehen sich auf Ampeln, Rot- und Grünphasen sowie Konflikte zwischen Fußgängern, Radfahrern und Verkehr.",
        "Typische Hinweise": "Ampel defekt, Rotphase zu lang, Querung unsicher"
    },
    10: {
        "Beschwerdethema": "Glasscherben und Verschmutzung an öffentlichen Orten",
        "Kurzbeschreibung": "Viele Meldungen betreffen Scherben und Verschmutzungen auf Spielplätzen, Radwegen, an Bushaltestellen oder auf öffentlichen Flächen.",
        "Typische Hinweise": "Glasscherben auf Spielplatz, Scherben auf Radweg, Müll an Haltestelle"
    }
}

# Prüfen, ob für jede Topic-Nummer eine Themenbezeichnung hinterlegt ist.
# Diese Prüfung bewertet nicht, ob die Bezeichnungen inhaltlich passend sind.
model_topics = set(nmf_summary["Topic"].astype(int))
label_topics = set(topic_meta.keys())

missing_labels = sorted(model_topics - label_topics)
unused_labels = sorted(label_topics - model_topics)

if missing_labels:
    raise ValueError(f"Für diese Topics fehlen Labels: {missing_labels}")

if unused_labels:
    print(f"Hinweis: Diese Labels werden aktuell nicht genutzt: {unused_labels}")


def top_child_categories(series, n=3):
    counts = series.value_counts().head(n)
    return "; ".join([f"{category} ({count})" for category, count in counts.items()])


# Kombinationen aus bereinigten Tokens sind nicht zwangsläufig Originalphrasen.
def top_ngrams_for_topic(texts, top_n=5):
    texts = [text for text in texts if isinstance(text, str) and len(text.split()) >= 2]

    if len(texts) == 0:
        return ""

    min_df_value = 2 if len(texts) >= 5 else 1

    vectorizer = CountVectorizer(
        ngram_range=(2, 3),
        min_df=min_df_value,
        token_pattern=r"(?u)\b[a-zäöüß][a-zäöüß]+\b"
    )

    try:
        X = vectorizer.fit_transform(texts)
    except ValueError:
        return ""

    terms = vectorizer.get_feature_names_out()
    scores = np.asarray(X.sum(axis=0)).ravel()

    top_indices = scores.argsort()[::-1][:top_n]
    return "; ".join([terms[i] for i in top_indices])


# Dominante Unterkategorien je Topic berechnen
child_summary = (
    df_topic_docs
    .groupby("NMF_Topic")["category_child"]
    .apply(top_child_categories)
    .reset_index()
    .rename(columns={
        "NMF_Topic": "Topic",
        "category_child": "Dominante Unterkategorien"
    })
)

# Häufige Kombinationen aus zwei oder drei Tokens im bereinigten Text zählen.
phrase_rows = []

for topic_id in sorted(model_topics):
    topic_texts = df_topic_docs.loc[
        df_topic_docs["NMF_Topic"] == topic_id,
        "text_clean"
    ].tolist()

    phrase_rows.append({
        "Topic": topic_id,
        "Auffällige Wortkombinationen": top_ngrams_for_topic(topic_texts, top_n=5)
    })

phrase_summary = pd.DataFrame(phrase_rows)

# Manuell hinterlegte Themenangaben in einen DataFrame überführen.
topic_meta_df = (
    pd.DataFrame.from_dict(topic_meta, orient="index")
    .rename_axis("Topic")
    .reset_index()
)

# Entscheidungsträger-Tabelle bauen
decision_table = (
    nmf_summary[["Topic", "Anzahl Meldungen", "Anteil Prozent"]]
    .merge(topic_meta_df, on="Topic", how="left")
    .merge(child_summary, on="Topic", how="left")
    .merge(phrase_summary, on="Topic", how="left")
)

decision_table = decision_table.sort_values(
    "Anzahl Meldungen",
    ascending=False
).reset_index(drop=True)

decision_table.insert(0, "Rang", np.arange(1, len(decision_table) + 1))

decision_table["Anteil"] = decision_table["Anteil Prozent"].map(
    lambda value: f"{value:.1f} %"
)

decision_table = decision_table[
    [
        "Rang",
        "Beschwerdethema",
        "Kurzbeschreibung",
        "Anzahl Meldungen",
        "Anteil",
        "Dominante Unterkategorien",
        "Typische Hinweise",
        "Auffällige Wortkombinationen"
    ]
]

display(decision_table)


In [ ]:
# Schritt 16 Übersicht für Entscheidungstragende aufbereiten

decision_table_final = decision_table.copy()

# Spalten für die Übersicht für Entscheidungstragende auswählen.
# Die zur Plausibilisierung verwendeten Kategorien bleiben in der technischen Übersicht.
decision_table_final = decision_table_final[
    [
        "Rang",
        "Beschwerdethema",
        "Kurzbeschreibung",
        "Anzahl Meldungen",
        "Anteil",
        "Typische Hinweise"
    ]
].copy()

# Ausgewählte Themenbezeichnungen an die bisherige inhaltliche Interpretation anpassen.
decision_table_final["Beschwerdethema"] = decision_table_final["Beschwerdethema"].replace({
    "Radwege, Entwässerung und Gefahrenstellen": "Gefahrenstellen auf Wegen und Radwegen",
    "Bäume, Äste und sonstige Hindernisse": "Hindernisse im öffentlichen Raum",
    "Glasscherben und Verschmutzung an öffentlichen Orten": "Scherben und Verschmutzung an öffentlichen Orten"
})

# Kurzbeschreibungen an die finalen Labels anpassen
decision_table_final.loc[
    decision_table_final["Beschwerdethema"] == "Gefahrenstellen auf Wegen und Radwegen",
    "Kurzbeschreibung"
] = "Mehrere Meldungen betreffen Gefahrenstellen auf Wegen und Radwegen, darunter Scherben, Wasser, verstopfte Abläufe oder Hindernisse."

decision_table_final.loc[
    decision_table_final["Beschwerdethema"] == "Hindernisse im öffentlichen Raum",
    "Kurzbeschreibung"
] = "Meldungen beschreiben blockierte oder eingeschränkte öffentliche Flächen, zum Beispiel durch Bäume, Äste, abgestellte Gegenstände oder andere Hindernisse."

decision_table_final.loc[
    decision_table_final["Beschwerdethema"] == "Scherben und Verschmutzung an öffentlichen Orten",
    "Kurzbeschreibung"
] = "Viele Meldungen betreffen Scherben und Verschmutzungen auf Spielplätzen, Radwegen, an Bushaltestellen oder auf öffentlichen Flächen."

# Textfelder bereinigen
decision_table_final["Typische Hinweise"] = decision_table_final["Typische Hinweise"].astype(str).str.strip()
decision_table_final["Anteil"] = decision_table_final["Anteil"].astype(str).str.strip()

# Spaltenname präzisieren
decision_table_final = decision_table_final.rename(
    columns={"Typische Hinweise": "Beispielhafte Hinweise"}
)

display(decision_table_final)


In [ ]:
# Schritt 17 Übersicht für Entscheidungstragende speichern

decision_table_final_path = OUTPUT_DIR / "entscheidungstraeger_themenuebersicht_final.csv"

decision_table_final.to_csv(
    decision_table_final_path,
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

print("Finale Entscheidungsträger-Übersicht gespeichert:")
print(decision_table_final_path.resolve())


In [ ]:
# Schritt 18 Themenhäufigkeiten als Balkendiagramm darstellen

import matplotlib.pyplot as plt

chart_df = decision_table_final.sort_values("Anzahl Meldungen", ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(chart_df["Beschwerdethema"], chart_df["Anzahl Meldungen"])
plt.xlabel("Anzahl Meldungen")
plt.ylabel("Beschwerdethema")
plt.title("Häufigste Beschwerdethemen im Mängelmelder Konstanz")
plt.tight_layout()

chart_path = OUTPUT_DIR / "haeufigste_beschwerdethemen_final.png"
plt.savefig(chart_path, dpi=150)
plt.show()

print("Diagramm gespeichert:")
print(chart_path.resolve())
